# **Start Section:**

In [1]:
# FIX K1: same notebook runs on Google Colab (Drive) or Kaggle (internet on).
# On Kaggle the working dir is /kaggle/working, so the original "./drive/MyDrive/..." paths
# below are reused unchanged; the dataset is copied there as train.csv / test.csv.
# The IBUG cells later change directory (%cd ibug, %cd .., %cd ..) and then use
# "./content/drive/..." and "/content/drive/..." (Colab layout); on Kaggle both are linked
# to /kaggle/working so every Excel file lands in the same output folder.
import os, glob, shutil
IN_KAGGLE = os.path.exists("/kaggle/working")
if IN_KAGGLE:
    _data_dir = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data"
    os.makedirs(_data_dir, exist_ok=True)
    shutil.copy(glob.glob("/kaggle/input/**/d09_a.csv", recursive=True)[0], f"{_data_dir}/train.csv")  # water_quality_ewqi train
    shutil.copy(glob.glob("/kaggle/input/**/d09_b.csv", recursive=True)[0], f"{_data_dir}/test.csv")   # water_quality_ewqi test
    for _link in ["/content", "/kaggle/content"]:
        if not os.path.exists(_link):
            os.symlink("/kaggle/working", _link)
    print("Kaggle: water_quality_ewqi data copied to", _data_dir)
else:
    from google.colab import drive
    drive.mount('/content/drive')


Kaggle: water_quality_ewqi data copied to ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data


In [2]:
# FIX K1: the treeffuser fork pins numpy<2 / scikit-learn==1.5.0 / lightgbm==4.3.0, which cannot be
# installed on Kaggle's Python 3.13. On Kaggle install the same code without those pins.
if IN_KAGGLE:
    !pip install -q --no-deps git+https://github.com/DaneshSelwal/treeffuser.git
    !pip install -q jaxtyping einops ml-collections
else:
    !pip install git+https://github.com/DaneshSelwal/treeffuser.git


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
treeffuser 0.1.3 requires jaxtyping<0.3.0,>=0.2.19, but you have jaxtyping 0.3.11 which is incompatible.
treeffuser 0.1.3 requires lightgbm==4.3.0, but you have lightgbm 4.6.0 which is incompatible.
treeffuser 0.1.3 requires ml-collections<0.2.0,>=0.1.1, but you have ml-collections 1.1.0 which is incompatible.
treeffuser 0.1.3 requires numpy<2.0,>=1.24, but you have numpy 2.1.3 which is incompatible.
treeffuser 0.1.3 requires scikit-learn==1.5.0, but you have scikit-learn 1.6.1 which is incompatible.


In [3]:
# FIX K1: original Colab install block kept as-is (runs only on Colab).
# On Kaggle: install only the packages the Kaggle image lacks, pinned to the Colab versions,
# and do NOT restart the kernel with os._exit(0).
if IN_KAGGLE:
    !pip install -q optuna==4.6.0 optunahub==0.4.0 cmaes==0.12.0 xgboost==3.1.2 catboost==1.2.8 gpboost==1.6.1 ngboost==0.5.8 pgbm==2.2.0 pytorch-tabnet2==4.5.3 interpret==0.7.4 lime==0.2.0.1 mapie==0.6.0 puncc==0.8.0 properscoring==0.1 XlsxWriter==3.2.9
else:
    import os
    os.environ["PIP_CONSTRAINT"] = "/tmp/numpy_constraint.txt"
    !echo "numpy==1.26.4" > /tmp/numpy_constraint.txt

    !pip uninstall -y torch torchaudio torchvision \
        torchao torchcodec torchdata torchtune torchsummary -q 2>/dev/null

    !pip uninstall -y tensorflow tensorflow-text tensorflow-hub tf-keras \
        tensorflow_decision_forests tensorflow-probability \
        tensorflow-Datasets tensorflow-metadata -q 2>/dev/null

    !pip uninstall -y numpy scikit-learn shap xgboost lightgbm dask \
        seaborn plotly openpyxl Cython catboost interpret lime -q 2>/dev/null

    !pip install numpy==1.26.4 -q
    !pip install scikit-learn==1.6.1 -q
    !pip install torch==2.9.0 -q

    !pip install lightgbm==4.6.0 -q
    !pip install xgboost==3.1.2 -q
    !pip install catboost==1.2.8 -q
    !pip install gpboost==1.6.1 -q
    !pip install ngboost==0.5.8 -q
    !pip install pgbm==2.2.0 -q
    !pip install pytorch-tabnet2==4.5.3 -q

    !pip install bayesian-optimization==3.2.0 -q
    !pip install optuna==4.6.0 -q
    !pip install optunahub==0.4.0 -q
    !pip install cmaes==0.12.0 -q

    !pip install shap==0.44.0 -q
    !pip install lime==0.2.0.1 -q
    !pip install interpret==0.7.4 -q

    !pip install mapie==0.6.0 -q
    !pip install puncc==0.8.0 -q
    !pip install skorch==1.3.1 -q
    !pip install properscoring==0.1 -q

    !pip install dask[dataframe]==2025.12.0 -q
    !pip install cython==3.0.12 -q
    !pip install seaborn==0.13.2 -q
    !pip install plotly==5.24.1 -q
    !pip install kaleido==1.2.0 -q
    !pip install openpyxl==3.1.5 -q
    !pip install XlsxWriter==3.2.9 -q
    !pip install cp==2020.12.3 -q

    !pip install numpy==1.26.4 --force-reinstall --no-deps -q

    os._exit(0)


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


  Installing build dependencies ... done


  Getting requirements to build wheel ... done


  Preparing metadata (pyproject.toml) ... done


# **Imports**

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import ngboost
from scipy.stats import randint
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.svm import SVR
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor
from gpboost import GPBoostRegressor
from ngboost import NGBRegressor
import optuna
import optunahub
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV
from interpret import show
from interpret.blackbox import LimeTabular, ShapKernel
from optuna.samplers import RandomSampler
import random
import time
from ngboost.distns import Normal
from ngboost.scores import LogScore
from scipy.stats import norm
from interpret import set_visualize_provider
from interpret.provider import InlineProvider
from interpret.glassbox import ExplainableBoostingRegressor
from interpret import show
import plotly.express as px
from io import BytesIO
from openpyxl import Workbook, load_workbook
import os
import properscoring as ps
import io
from openpyxl.drawing.image import Image as openpyxlImage
import warnings
import xlsxwriter
from pgbm.sklearn import HistGradientBoostingRegressor
import torch
from pgbm.torch import PGBM
import plotly.graph_objects as go
warnings.filterwarnings('ignore')
import pickle
import json
from mapie.subsample import Subsample
from mapie.regression import MapieRegressor
from deel.puncc.metrics import regression_sharpness, regression_mean_coverage
from deel.puncc.api.prediction import BasePredictor, DualPredictor
from deel.puncc.regression import SplitCP, CVPlus, CQR
from deel.puncc.plotting import plot_prediction_intervals
from sklearn.model_selection import train_test_split
from typing_extensions import TypedDict
from typing import Union
from mapie.metrics import regression_coverage_score
from sklearn.model_selection import KFold
from PIL import Image as PImage
from pathlib import Path
from scipy.stats import norm, kstest
from pytorch_tabnet import TabNetRegressor
from treeffuser import Treeffuser, Samples
from treeffuser._score_models import TreeBasedScoreModel

[1/2] c++ -MMD -MF splitgain_cpu.o.d -DTORCH_EXTENSION_NAME=split_decision -DTORCH_API_INCLUDE_EXTENSION_H -isystem /usr/local/lib/python3.13/dist-packages/torch/include -isystem /usr/local/lib/python3.13/dist-packages/torch/include/torch/csrc/api/include -isystem /usr/include/python3.13 -fPIC -std=c++17 -c /usr/local/lib/python3.13/dist-packages/pgbm/torch/splitgain_cpu.cpp -o splitgain_cpu.o 


[2/2] c++ splitgain_cpu.o -shared -L/usr/local/lib/python3.13/dist-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o split_decision.so


I0000 00:00:1795364571.564976      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


I0000 00:00:1795364589.475299      33 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [5]:
# Go to find & replace button and replace (water_quality_ewqi_uncertainty_analysis) with your folder name. Rename your train and test Dataset as train.csv and test.csv.
# Modify the names of the feature in the below cell.
# Replace (str) with actual data label name.

In [6]:
feature_names = ['pH', 'TDS', 'EC', 'TH', 'ca', 'mg', 'CO3', 'HCO3', 'NA', 'K', 'CHLORIDE', 'SULPHATE', 'NITRATE', 'FLUORIDE']
TARGET_LABEL = 'Pre-monsoon EWQI'  # FIX B12: target name used in plot labels (was 'str', the concrete dataset's target, on every dataset)

In [7]:
train_data_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/train.csv"
test_data_path = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/data/test.csv"
train_data = pd.read_csv(train_data_path)
test_data = pd.read_csv(test_data_path)
print("Training data loaded successfully.")
print("Test data loaded successfully.")

Training data loaded successfully.
Test data loaded successfully.


In [8]:
print("\nShape of training data:", train_data.shape)
print("First 5 rows of training data:\n", train_data.head(5))
print("\nShape of test data:", test_data.shape)
print("First 5 rows of test data:\n", test_data.head(5))


Shape of training data: (47, 15)
First 5 rows of training data:
      pH   TDS    EC    TH   ca    mg  CO3  HCO3   NA    K  CHLORIDE  SULPHATE  \
0  7.90   552   865   452  272   180  0.2   4.5  260   44       375       681   
1  7.52  1985  2256  1054  529   525  0.0  15.0  272   47       728       324   
2  7.65  1865  2032   636  371   265  0.0   9.0  373   25       713       503   
3  7.25  4855  7856  2050  795  1255  0.0  12.5  335  106        81       676   
4  7.96  2855  3696   865  310   555  0.3   4.6  184  106       698       324   

   NITRATE  FLUORIDE  Pre monsoon EWQI   
0       60      0.40              70.52  
1       10      0.65             138.78  
2       54      0.85             113.07  
3       17      0.69             266.32  
4       25      1.20             158.05  

Shape of test data: (47, 15)
First 5 rows of test data:
      pH   TDS    EC    TH     ca      mg  CO3  HCO3   NA    K  CHLORIDE  \
0  8.38   625   975   320  120.0   200.0  0.0   5.8  280   35 

In [9]:
X_train = train_data.iloc[:, :-1]
y_train = train_data.iloc[:, -1]
X_test = test_data.iloc[:, :-1]
y_test = test_data.iloc[:, -1]
# FIX D4 / D5 (approved 2026-11-22): settings that follow the test-set size
N_BINS = max(2, min(10, len(y_test) // 10))  # D4: bins for binned metrics (was 10 on every dataset; pile: 2-3 points per bin)
COP_T_BURNIN = min(50, len(y_test) // 2)  # D5: COP burn-in (was 50, so with < 50 test points COP's correction never started)
x_test= X_test
print("\nShape of X_train:", X_train.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (47, 14)
Shape of y_train: (47,)
Shape of X_test: (47, 14)
Shape of y_test: (47,)


In [10]:
# Apply z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Print the first five rows of the normalized data
print("\nFirst five rows of normalized X_train:")
print(X_train[:5])

print("\nFirst five rows of normalized X_test:")
print(X_test[:5])


First five rows of normalized X_train:
[[ 0.3797749  -1.15230942 -0.94309903 -0.47923131 -0.09229732 -0.56852482
  -0.03922784 -1.18768739 -0.02713192 -0.87884263 -0.2770724   1.15873613
   1.41276153 -1.25316406]
 [-0.78765707 -0.2953837  -0.52059702  0.45803078  1.31357822  0.10564733
  -0.50015494  1.61405127  0.07159325 -0.77136925  1.29937074 -0.45273785
  -1.61366471 -0.53733141]
 [-0.38827245 -0.36714301 -0.58863473 -0.19275917  0.44926563 -0.40242444
  -0.50015494  0.01305775  0.90253007 -1.55950739  1.23238307  0.35525611
   1.04959038  0.03533472]
 [-1.61714822  1.42085972  1.18034563  2.00871688  2.76868675  1.53215652
  -0.50015494  0.94697063  0.58990037  1.34227396 -1.59003071  1.13616647
  -1.18996504 -0.42279818]
 [ 0.56410627  0.22487128 -0.08321177  0.16377408  0.11557533  0.16427099
   0.19123571 -1.16100417 -0.6523913   1.34227396  1.1653954  -0.45273785
  -0.70573684  1.03750043]]

First five rows of normalized X_test:
[[ 1.85442582 -1.10865585 -0.90968766 -0.6847

# **Functions:**

In [11]:
# Define the model classes
model_classes = {
    'Random Forest': RandomForestRegressor,
    'Gradient Boosting': GradientBoostingRegressor,
    'XGBoost': XGBRegressor,
    'LightGBM': LGBMRegressor,
    'GPBoost': GPBoostRegressor,
    'CatBoost': CatBoostRegressor,
    'HistGradientBoosting': HistGradientBoostingRegressor,
    'TabNet': TabNetRegressor,
    'NGBoost': NGBRegressor
}

def _ensure_excel_file(path):
    import os
    import pandas as pd
    if not os.path.exists(path):
        parent = os.path.dirname(path)
        if parent:
            os.makedirs(parent, exist_ok=True)
        pd.DataFrame().to_excel(path)
    return path

In [12]:
def get_best_model_params(results, model_name):
    # Map model names to dictionary keys, assuming keys are strings like 'XGBoost' and not objects
    model_keys = {
        'LightGBM': 'LightGBM',
        'XGBoost': 'XGBoost',
        'GPBoost': 'GPBoost',
        'NGBoost': 'NGBoost',
        'GradientBoosting': 'Gradient Boosting',
        'HGBR' : 'HistGradientBoosting',
        'CatBoost' : 'CatBoost',
        'PGBM' : 'PGBM'
    }

    # Ensure the requested model name is valid
    if model_name not in model_keys:
        raise ValueError(f"Model name '{model_name}' is not recognized. Available models are: {list(model_keys.keys())}")

    # Filter out entries for the specified model
    model_entries = {key: value for key, value in results.items() if key[0] == model_keys[model_name]}

    # Find the entry with the best (lowest) 'best_score'
    best_entry_key, best_entry_value = min(model_entries.items(), key=lambda item: item[1]['best_score'])

    # Return the best hyperparameters
    return best_entry_value['best_params']

In [13]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_plots(predictions_df, y_test, excel_path):
    # 1. Predicted vs. Actual Plot
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x='Actual', y='Mean', data=predictions_df)
    plt.plot([predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             [predictions_df['Actual'].min(), predictions_df['Actual'].max()],
             color='red', linestyle='--')
    plt.title(f'Predicted {TARGET_LABEL} vs. Actual {TARGET_LABEL}')
    plt.xlabel(f'Actual {TARGET_LABEL}')
    plt.ylabel(f'Predicted Mean {TARGET_LABEL}')
    plt.tight_layout()
    predicted_vs_actual_path = 'predicted_vs_actual.png'
    _ensure_parent_dir(predicted_vs_actual_path)
    plt.savefig(_ensure_parent_dir(predicted_vs_actual_path))
    plt.close()

    # 2. Uncertainty Bands
    plt.figure(figsize=(12, 8))
    plt.errorbar(x=range(len(predictions_df)), y=predictions_df['Mean'],
                 yerr=predictions_df['StdDev'], fmt='o', markersize=4, ecolor='red', elinewidth=1, capsize=5, alpha=0.7)
    plt.plot(range(len(predictions_df)), predictions_df['Mean'], color='black', linewidth=0.5, alpha=0.7)
    plt.title('Predicted Mean with Uncertainty Bands', fontsize=16)
    plt.xlabel('Sample Number', fontsize=14)
    plt.ylabel(f'Predicted Mean {TARGET_LABEL}')
    plt.tight_layout()
    uncertainty_bands_path = 'uncertainty_bands.png'
    _ensure_parent_dir(uncertainty_bands_path)
    plt.savefig(_ensure_parent_dir(uncertainty_bands_path))
    plt.close()

    # 3. Histogram of Standard Deviations
    plt.figure(figsize=(12, 8))
    sns.histplot(predictions_df['StdDev'], bins=20, kde=True)
    plt.title('Distribution of Predicted Standard Deviations')
    plt.xlabel('Standard Deviation')
    plt.ylabel('Frequency')
    plt.tight_layout()
    histogram_stddev_path = 'histogram_stddev.png'
    _ensure_parent_dir(histogram_stddev_path)
    plt.savefig(_ensure_parent_dir(histogram_stddev_path))
    plt.close()

    # 4. Residual Plot
    residuals = predictions_df['Actual'] - predictions_df['Mean']
    plt.figure(figsize=(12, 8))
    sns.scatterplot(x=predictions_df['Mean'], y=residuals)
    plt.axhline(0, color='red', linestyle='--')
    plt.title('Residual Plot')
    plt.xlabel(f'Predicted Mean {TARGET_LABEL}')
    plt.ylabel('Residuals')
    plt.tight_layout()
    residual_plot_path = 'residual_plot.png'
    _ensure_parent_dir(residual_plot_path)
    plt.savefig(_ensure_parent_dir(residual_plot_path))
    plt.close()

    # Plot the standard deviation of the predictions
    plt.figure(figsize=(12, 6))
    plt.plot(
        predictions_df['StdDev'],
        label='Standard Deviation',
        color='green',
        marker='o',
        linestyle='dashed'
    )
    plt.xlabel('Sample Number')
    plt.ylabel('Standard Deviation')
    plt.title('Standard Deviation of Predictions')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    stddev_plot_path = 'stddev_plot.png'
    _ensure_parent_dir(stddev_plot_path)
    plt.savefig(_ensure_parent_dir(stddev_plot_path))
    plt.close()

    # Plot the actual vs predicted mean with confidence intervals
    plt.figure(figsize=(12, 6))
    plt.plot(
        y_test,
        label=f'Actual {TARGET_LABEL}',
        color='black',
        marker='o',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    plt.plot(
        predictions_df['Mean'],
        label=f'Predicted Mean {TARGET_LABEL}',
        color='red',
        marker='x',
        linestyle='dashed',
        linewidth=1,
        markersize=4
    )
    confidence_interval = 1.96 * predictions_df['StdDev']
    plt.fill_between(
        np.arange(len(predictions_df)),
        predictions_df['Mean'] - confidence_interval,
        predictions_df['Mean'] + confidence_interval,
        color='blue',
        alpha=0.2,
        label='95% Confidence Interval'
    )
    plt.xlabel('Sample Number')
    plt.ylabel(f' {TARGET_LABEL}')
    plt.title('Predictions with Confidence Intervals')
    plt.legend(loc='upper left', fontsize='small', bbox_to_anchor=(1.05, 1), borderaxespad=0.)
    plt.tight_layout()
    prediction_plot_path = 'prediction_plot.png'
    _ensure_parent_dir(prediction_plot_path)
    plt.savefig(_ensure_parent_dir(prediction_plot_path))
    plt.close()

    # Create a Pandas Excel writer using XlsxWriter as the engine.
    writer = pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter')

    # Write the dataframe to a worksheet.
    writer
    predictions_df.to_excel(writer, sheet_name='Predictions', index=False)

    # Access the XlsxWriter workbook and worksheet objects from the writer object.
    workbook = writer.book

    # Insert the plots into new worksheets.
    worksheet_predicted_vs_actual = workbook.add_worksheet('Predicted vs Actual')
    worksheet_predicted_vs_actual.insert_image('A1', predicted_vs_actual_path)

    worksheet_uncertainty_bands = workbook.add_worksheet('Uncertainty Bands')
    worksheet_uncertainty_bands.insert_image('A1', uncertainty_bands_path)

    worksheet_histogram_stddev = workbook.add_worksheet('Histogram StdDev')
    worksheet_histogram_stddev.insert_image('A1', histogram_stddev_path)

    worksheet_residual_plot = workbook.add_worksheet('Residual Plot')
    worksheet_residual_plot.insert_image('A1', residual_plot_path)

    worksheet_stddev = workbook.add_worksheet('StdDev Plot')
    worksheet_stddev.insert_image('A1', stddev_plot_path)

    worksheet_prediction = workbook.add_worksheet('Predictions Plot')
    worksheet_prediction.insert_image('A1', prediction_plot_path)

    # Close the Pandas Excel writer and output the Excel file.
    writer.close()
    print(f"Excel file saved at {excel_path}")

    # Optionally, remove the image files now that they're embedded in the Excel file
    if os.path.exists(str(predicted_vs_actual_path)): os.remove(str(predicted_vs_actual_path))
    if os.path.exists(str(uncertainty_bands_path)): os.remove(str(uncertainty_bands_path))
    if os.path.exists(str(histogram_stddev_path)): os.remove(str(histogram_stddev_path))
    if os.path.exists(str(residual_plot_path)): os.remove(str(residual_plot_path))
    if os.path.exists(str(stddev_plot_path)): os.remove(str(stddev_plot_path))
    if os.path.exists(str(prediction_plot_path)): os.remove(str(prediction_plot_path))

In [14]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_crps_decomposition(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=N_BINS
):

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    # -------------------------------
    # Total CRPS
    # -------------------------------
    crps_values = ps.crps_gaussian(
        y_true, mu=mu_pred, sig=sigma_pred
    )
    total_crps = np.mean(crps_values)

    # -------------------------------
    # FIX P1: Reliability / Resolution / Uncertainty from the Hersbach (2000) decomposition,
    # "Decomposition of the CRPS for ensemble prediction systems", Weather and Forecasting 15.
    # Was: Reliability = mean of the model's own CRPS over 10 groups (= total CRPS), so
    # Resolution always came out = Uncertainty and the split carried no information.
    # Each predicted Gaussian is represented by M members at its quantile levels (k-0.5)/M.
    #   CRPS_ens = Reliability + CRPS_pot,  CRPS_pot = Uncertainty - Resolution
    # -------------------------------
    M = 100
    sig_safe = np.clip(sigma_pred, 1e-12, None)
    levels = (np.arange(1, M + 1) - 0.5) / M
    ens = np.sort(norm.ppf(levels[None, :], loc=mu_pred[:, None], scale=sig_safe[:, None]), axis=1)  # (n, M)
    yy = y_true[:, None]
    n_obs = len(y_true)

    # alpha_i, beta_i for the inner bins [x_i, x_{i+1}], i = 1..M-1 (Hersbach eq. 26)
    lo, hi = ens[:, :-1], ens[:, 1:]
    alpha_in = np.where(yy > hi, hi - lo, np.where(yy > lo, yy - lo, 0.0))
    beta_in = np.where(yy < lo, hi - lo, np.where(yy < hi, hi - yy, 0.0))
    alpha_bar = np.zeros(M + 1)
    beta_bar = np.zeros(M + 1)
    alpha_bar[1:M] = alpha_in.mean(axis=0)
    beta_bar[1:M] = beta_in.mean(axis=0)
    # outliers (Hersbach eq. 27): below the lowest / above the highest member
    beta_bar[0] = np.mean(np.where(y_true < ens[:, 0], ens[:, 0] - y_true, 0.0))
    alpha_bar[M] = np.mean(np.where(y_true > ens[:, -1], y_true - ens[:, -1], 0.0))

    p = np.arange(M + 1) / M
    g = alpha_bar + beta_bar
    o = np.divide(beta_bar, g, out=np.zeros_like(g), where=g > 0)
    # outlier bins: o_0 = freq(y < x_1), o_M = 1 - freq(y > x_M) (Hersbach eq. 33)
    o[0] = np.mean(y_true < ens[:, 0])
    o[M] = 1.0 - np.mean(y_true > ens[:, -1])
    g[0] = beta_bar[0] / o[0] if o[0] > 0 else 0.0
    g[M] = alpha_bar[M] / (1.0 - o[M]) if o[M] < 1 else 0.0

    reliability = float(np.sum(g * (o - p) ** 2))
    crps_pot = float(np.sum(g * o * (1.0 - o)))
    # Uncertainty = CRPS of the sample climatology of the observations
    uncertainty = float(np.abs(y_true[:, None] - y_true[None, :]).sum() / (2.0 * n_obs ** 2))
    resolution = uncertainty - crps_pot

    # -------------------------------
    # Per-bin table (unchanged): test rows sorted by predicted mean, split into n_bins groups
    # -------------------------------
    sort_idx = np.argsort(mu_pred)
    y_sorted = y_true[sort_idx]
    mu_sorted = mu_pred[sort_idx]
    sigma_sorted = sigma_pred[sort_idx]

    bins = np.array_split(np.arange(len(y_true)), n_bins)

    bin_results = []

    for i, b in enumerate(bins):
        bin_y = y_sorted[b]
        bin_mu = mu_sorted[b]
        bin_sigma = sigma_sorted[b]

        bin_crps = np.mean(
            ps.crps_gaussian(bin_y, mu=bin_mu, sig=bin_sigma)
        )

        bin_results.append({
            "Bin": i + 1,
            "Bin_Size": len(b),
            "Mean_Pred_Mu": np.mean(bin_mu),
            "Mean_Pred_Sigma": np.mean(bin_sigma),
            "Bin_CRPS": bin_crps
        })

    # -------------------------------
    # Summary table
    # -------------------------------
    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Total_CRPS": total_crps,
        "Reliability": reliability,
        "Resolution": resolution,
        "Uncertainty": uncertainty,
        "Num_Bins": n_bins
    }])

    bin_df = pd.DataFrame(bin_results)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_summary = f"CRPS_Summary_{model_name}"
    sheet_bins = f"CRPS_Bins_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)
        writer
        bin_df.to_excel(writer, sheet_name=sheet_bins, index=False)

    print(f"✅ CRPS decomposition saved to:\n{excel_path}")
    print(f"   Sheets: {sheet_summary}, {sheet_bins}")

    return summary_df

In [15]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_pit_results(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_bins=N_BINS
):
    """
    Generate PIT values, histogram, calibration diagnostics,
    and save everything to Excel with the PIT plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()

    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Compute PIT values
    # -------------------------------
    pit_values = norm.cdf(y_true, loc=mu_pred, scale=sigma_pred)

    pit_df = pd.DataFrame({"PIT_Value": pit_values})

    # -------------------------------
    # Histogram data
    # -------------------------------
    hist_density, bin_edges = np.histogram(
        pit_values, bins=n_bins, range=(0, 1), density=True
    )
    bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    hist_df = pd.DataFrame({
        "Bin_Center": bin_centers,
        "Density": hist_density
    })

    # -------------------------------
    # Calibration diagnostics
    # -------------------------------
    pit_mean = pit_values.mean()
    pit_var = pit_values.var()
    ks_stat, ks_pvalue = kstest(pit_values, 'uniform')

    if ks_pvalue > 0.05:
        diagnosis = "Well-calibrated (Uniform PIT)"
    elif pit_var < 1/12:
        diagnosis = "Over-dispersed (Under-confident)"
    elif pit_var > 1/12:
        diagnosis = "Under-dispersed (Over-confident)"
    else:
        diagnosis = "Biased or miscalibrated"

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_PIT": pit_mean,
        "Variance_PIT": pit_var,
        "Ideal_Variance_(1/12)": 1/12,
        "KS_Statistic": ks_stat,
        "KS_pvalue": ks_pvalue,
        "Calibration_Diagnosis": diagnosis
    }])

    # -------------------------------
    # Create PIT plot in memory
    # -------------------------------
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(
        pit_values,
        bins=n_bins,
        density=True,
        color="skyblue",
        edgecolor="black",
        alpha=0.75
    )
    ax.axhline(1, color="red", linestyle="--", label="Perfect Calibration")
    ax.set_xlabel("PIT Value")
    ax.set_ylabel("Density")
    ax.set_title(f"PIT Histogram – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_values = f"PIT_Values_{model_name}"
    sheet_hist = f"PIT_Histogram_{model_name}"
    sheet_summary = f"PIT_Summary_{model_name}"
    sheet_plot = f"PIT_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        pit_df.to_excel(writer, sheet_name=sheet_values, index=False)
        writer
        hist_df.to_excel(writer, sheet_name=sheet_hist, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        # Create empty sheet for plot
        workbook = writer.book
        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print(f"✅ PIT values, histogram, summary, and EMBEDDED plot saved to:")
    print(f"   {excel_path}")

    return pit_df

In [16]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
def generate_and_save_calibration_curve(
    predictions_df,
    y_true,
    excel_path,
    model_name="Model",
    n_quantiles=10
):
    """
    Generate quantile calibration curve (reliability diagram),
    compute calibration errors, and save everything to Excel
    with the plot EMBEDDED.
    """

    # -------------------------------
    # Input handling
    # -------------------------------
    mu_pred = predictions_df['Mean'].values
    sigma_pred = predictions_df['StdDev'].values

    if 'Actual' in predictions_df.columns:
        y_true = predictions_df['Actual'].values
    else:
        y_true = np.asarray(y_true).ravel()

    y_true = np.asarray(y_true).ravel()
    sigma_pred = np.clip(sigma_pred, 1e-6, None)

    # -------------------------------
    # Quantiles
    # -------------------------------
    quantiles = np.linspace(0, 1, n_quantiles + 1)

    observed_freq = []
    predicted_q = []

    for q in quantiles:
        # Predicted value at quantile q
        thresh = norm.ppf(q, loc=mu_pred, scale=sigma_pred)

        # Observed frequency
        obs = np.mean(y_true <= thresh)

        predicted_q.append(q)
        observed_freq.append(obs)

    predicted_q = np.array(predicted_q)
    observed_freq = np.array(observed_freq)

    # -------------------------------
    # Calibration metrics
    # -------------------------------
    mce = np.mean(np.abs(observed_freq - predicted_q))
    rmsce = np.sqrt(np.mean((observed_freq - predicted_q) ** 2))

    calib_df = pd.DataFrame({
        "Predicted_Quantile": predicted_q,
        "Observed_Frequency": observed_freq,
        "Absolute_Error": np.abs(observed_freq - predicted_q)
    })

    summary_df = pd.DataFrame([{
        "Model": model_name,
        "Mean_Calibration_Error (MCE)": mce,
        "RMS_Calibration_Error (RMSCE)": rmsce,
        "Num_Quantiles": n_quantiles
    }])

    # -------------------------------
    # Create calibration plot
    # -------------------------------
    fig, ax = plt.subplots(figsize=(7, 7))

    ax.plot([0, 1], [0, 1], 'k--', label="Perfect Calibration")
    ax.plot(
        predicted_q,
        observed_freq,
        'o-',
        color='teal',
        label=f"{model_name}"
    )

    ax.set_xlabel("Predicted Quantile (Confidence Level)")
    ax.set_ylabel("Observed Frequency")
    ax.set_title(f"Quantile Calibration Curve – {model_name}")
    ax.legend()
    fig.tight_layout()

    img_buffer = BytesIO()
    fig.savefig(img_buffer, format="png", dpi=300)
    plt.close(fig)
    img_buffer.seek(0)

    # -------------------------------
    # Save to Excel
    # -------------------------------
    excel_path = Path(excel_path)
    excel_path.parent.mkdir(parents=True, exist_ok=True)

    sheet_curve = f"Calibration_Curve_{model_name}"
    sheet_summary = f"Calibration_Summary_{model_name}"
    sheet_plot = f"Calibration_Plot_{model_name}"

    with pd.ExcelWriter(_ensure_parent_dir(_ensure_excel_file(
        excel_path)),
        engine="openpyxl",
        mode="a" if excel_path.exists() else "w",
        if_sheet_exists="replace"
    ) as writer:

        writer
        calib_df.to_excel(writer, sheet_name=sheet_curve, index=False)
        writer
        summary_df.to_excel(writer, sheet_name=sheet_summary, index=False)

        workbook = writer.book

        worksheet = workbook.create_sheet(sheet_plot)

        img = openpyxlImage(img_buffer)
        img.width = img.width * 0.4
        img.height = img.height * 0.4

        img.anchor = "B2"
        worksheet.add_image(img)

    print("✅ Calibration curve & metrics saved to Excel:")
    print(f"   {excel_path}")

    return calib_df

In [17]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

import os
# Define the path to the folder
folder_path = "/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis"

# Create the folder if it doesn't exist
os.makedirs(folder_path, exist_ok=True)

def plot_metrics_and_save_to_excel(writer, predictions_df, y_true, model_name):
    # Calculate metrics
    mean_pred = predictions_df['Mean']
    std_pred = predictions_df['StdDev']

    # Sharpness is the standard deviation
    sharpness = std_pred.mean()

    # CRPS for each prediction
    crps_values = ps.crps_gaussian(y_true, mean_pred, std_pred)

    # Negative Log Likelihood (NLL) for each prediction
    nll_values = -norm.logpdf(y_true, loc=mean_pred, scale=std_pred)
    nll = nll_values.mean()

    # Calibration (Validity)
    percentiles = np.linspace(0, 100, 101)
    calibration = [np.mean((y_true <= np.percentile(mean_pred + std_pred * np.random.randn(*mean_pred.shape), p))) for p in percentiles]

    # Create a DataFrame to store the metrics
    metrics_df = pd.DataFrame({
        'Mean Prediction': mean_pred,
        'Standard Deviation': std_pred,
        'CRPS': crps_values,
        'NLL': nll_values
    })

    # Add summary statistics
    summary_df = pd.DataFrame({
        'Metric': ['Sharpness', 'Mean CRPS', 'Mean NLL'],
        'Value': [sharpness, crps_values.mean(), nll]
    })

    # Write the metrics and summary to the Excel sheet
    metrics_sheet_name = f'{model_name} Metrics'
    writer
    metrics_df.to_excel(writer, sheet_name=metrics_sheet_name, index=False)

    summary_sheet_name = f'{model_name} Summary'
    writer
    summary_df.to_excel(writer, sheet_name=summary_sheet_name, index=False)

    # Plotting
    fig, axs = plt.subplots(2, 2, figsize=(12, 10))

    # Sharpness plot
    axs[0, 0].hist(std_pred, bins=30, color='skyblue', edgecolor='black')
    axs[0, 0].set_title(f'Sharpness (StdDev) for {model_name}')
    axs[0, 0].set_xlabel('Standard Deviation')
    axs[0, 0].set_ylabel('Frequency')

    # CRPS plot (distribution)
    axs[0, 1].hist(crps_values, bins=30, color='lightgreen', edgecolor='black')
    axs[0, 1].set_title(f'CRPS Distribution for {model_name}')
    axs[0, 1].set_xlabel('CRPS')
    axs[0, 1].set_ylabel('Frequency')

    # NLL plot (distribution)
    axs[1, 0].hist(nll_values, bins=30, color='salmon', edgecolor='black')
    axs[1, 0].set_title(f'NLL Distribution for {model_name}')
    axs[1, 0].set_xlabel('NLL')
    axs[1, 0].set_ylabel('Frequency')

    # Calibration plot
    axs[1, 1].plot(percentiles, calibration, label='Calibration Curve')
    axs[1, 1].plot(percentiles, percentiles / 100, linestyle='--', label='Perfect Calibration')
    axs[1, 1].set_title(f'Calibration (Validity) for {model_name}')
    axs[1, 1].set_xlabel('Predicted Percentile')
    axs[1, 1].set_ylabel('Observed Frequency')
    axs[1, 1].legend()

    plt.tight_layout()

    # Save plots to images
    plot_image_path = f'{model_name}_plots.png'
    fig.savefig(_ensure_parent_dir(plot_image_path))
    plt.close(fig)

    # Insert the plot image into the Excel sheet
    sheet_name = f'{model_name} Plots'
    if sheet_name not in writer.sheets:  # Check if sheet already exists
        # If sheet doesn't exist, create a new one
        worksheet = writer.book.add_worksheet(sheet_name)
    else:
        # If sheet exists, get the existing sheet object to avoid duplicates
        worksheet = writer.sheets[sheet_name]
    worksheet.insert_image('A1', plot_image_path)

# **Tuned Parameters**

In [18]:
# FIX K2: load the tuning results saved by hyperparameter_tuning/Optuna_autosampler.ipynb
# (was: a hand-pasted copy of an older tuning run that had been selected on the TEST set).
import pickle
if IN_KAGGLE:
    _bp = glob.glob("/kaggle/input/**/hyperparameter_tuning/best_params.pkl", recursive=True)[0]
else:
    _bp = "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/hyperparameter_tuning/best_params.pkl"
with open(_bp, "rb") as f:
    best_scores_autosampler = pickle.load(f)
print("Loaded tuning results from", _bp, "-", len(best_scores_autosampler), "model x pruner entries")


Loaded tuning results from /kaggle/input/datasets/daneshselwal/water-quality-ewqi-tuned-params/hyperparameter_tuning/best_params.pkl - 80 model x pruner entries


In [19]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 815.3942506476026,
  'cv_rmse': 28.555109011306584,
  'best_params': {'n_estimators': 200,
   'criterion': 'squared_error',
   'max_depth': 20,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.5,
   'max_leaf_nodes': 200,
   'min_impurity_decrease': 0.1,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.1},
  'test_mse': 517.6088147057782,
  'test_rmse': np.float64(22.751017882850388),
  'test_corr_coef': np.float64(0.9547680128224817),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 800.6499765894229,
  'cv_rmse': 28.295758985922657,
  'best_params': {'n_estimators': 300,
   'criterion': 'squared_error',
   'max_depth': 20,
   'min_samples_split': 0.01,
   'min_samples_leaf': 0.01,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.5,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 

In [20]:
best_scores_autosampler

{('Random Forest', 'MedianPruner'): {'best_score': 815.3942506476026,
  'cv_rmse': 28.555109011306584,
  'best_params': {'n_estimators': 200,
   'criterion': 'squared_error',
   'max_depth': 20,
   'min_samples_split': 0.01,
   'min_samples_leaf': 1,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.5,
   'max_leaf_nodes': 200,
   'min_impurity_decrease': 0.1,
   'n_jobs': -1,
   'random_state': 42,
   'verbose': 0,
   'warm_start': False,
   'ccp_alpha': 0.1},
  'test_mse': 517.6088147057782,
  'test_rmse': np.float64(22.751017882850388),
  'test_corr_coef': np.float64(0.9547680128224817),
  'pruner': 'MedianPruner'},
 ('Random Forest', 'NopPruner'): {'best_score': 800.6499765894229,
  'cv_rmse': 28.295758985922657,
  'best_params': {'n_estimators': 300,
   'criterion': 'squared_error',
   'max_depth': 20,
   'min_samples_split': 0.01,
   'min_samples_leaf': 0.01,
   'min_weight_fraction_leaf': 0.0,
   'max_features': 0.5,
   'max_leaf_nodes': None,
   'min_impurity_decrease': 

# **Probabilistic distributions using NGBoost**

In [21]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'NGBoost')

# Remove any conflicting parameters from best_params
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize NGBoost model with best_params
ngb_model = NGBRegressor(Dist=Normal, **best_params)

# Fit the model
ngb_model.fit(X_train, y_train)

# Predict the distribution
pred_dist = ngb_model.pred_dist(X_test)

# Extract mean and standard deviation
predictions_NGB_df = pd.DataFrame({
    'Mean': pred_dist.loc,
    'StdDev': pred_dist.scale,
    'Actual': y_test.ravel()
})

# Calculate and print the mean squared error
mse = mean_squared_error(y_test, pred_dist.loc)
print(f"Mean Squared Error: {mse:.2f}")

# Print the DataFrame with mean and standard deviation
print(predictions_NGB_df.head())

Mean Squared Error: 472.28
         Mean    StdDev  Actual
0   78.276032  0.000007   73.07
1  140.897658  0.000001  141.67
2   88.697166  0.000004  110.44
3  268.312806  0.000003  275.01
4  136.794993  0.000011  202.57


In [22]:
generate_and_save_plots(predictions_NGB_df, y_test, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


In [23]:
generate_and_save_crps_decomposition(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx
   Sheets: CRPS_Summary_NGBoost, CRPS_Bins_NGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,NGBoost,16.550005,11.354742,24.924383,30.119647,4


In [24]:
generate_and_save_pit_results(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


,PIT_Value
0,0.0
1,1.0
2,1.0
3,1.0
4,1.0
5,1.0
6,1.0
7,1.0
8,0.0
9,1.0


In [25]:
generate_and_save_calibration_curve(
    predictions_NGB_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx",
    model_name="NGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/NGBoost_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.234043,0.134043
2,0.2,0.234043,0.034043
3,0.3,0.234043,0.065957
4,0.4,0.234043,0.165957
5,0.5,0.234043,0.265957
6,0.6,0.234043,0.365957
7,0.7,0.234043,0.465957
8,0.8,0.234043,0.565957
9,0.9,0.234043,0.665957


# **Probabilistic distributions using PGBM**

In [26]:
# Assuming best_params is obtained correctly
best_params = get_best_model_params(best_scores_autosampler, 'PGBM')

# Remove any conflicting parameters from best_params
# Note: Adjust the incompatible keys based on PGBM's requirements
incompatible_keys = ['Dist', 'Score']
for key in incompatible_keys:
    best_params.pop(key, None)

# Initialize PGBM model
pgbm_model = PGBM()

def mseloss_objective(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    gradient = yhat - y
    hessian = torch.ones_like(yhat)
    return gradient, hessian

def rmseloss_metric(yhat, y, sample_weight=None):
    # Ensure that yhat and y are PyTorch tensors
    if not torch.is_tensor(yhat):
        yhat = torch.from_numpy(np.array(yhat)).float()
    if not torch.is_tensor(y):
        y = torch.from_numpy(np.array(y)).float()
    loss = torch.sqrt(torch.mean((yhat - y) ** 2))
    return loss

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)
# Fit the model
pgbm_model.train((X_train, y_train), objective=mseloss_objective, metric=rmseloss_metric, params = best_params)

# Predict the distribution
pred_dist = pgbm_model.predict_dist(X_test)

# Extract mean and standard deviation
# Note: PGBM might return predictions in a different format, adjust accordingly
predictions_PGBM_df = pd.DataFrame({
    'Mean': pred_dist.mean(axis=0),
    'StdDev': pred_dist.std(axis=0),
    'Actual': y_test.ravel()
})

# Calculate and print the mean squared error
mse = mean_squared_error(y_test, predictions_PGBM_df['Mean'])
print(f"Mean Squared Error: {mse:.2f}")

# Print the DataFrame with mean and standard deviation
print(predictions_PGBM_df.head())

Training on CPU


Estimator 0/300, Train metric: 55.0218
Estimator 1/300, Train metric: 53.3739
Estimator 2/300, Train metric: 52.0749
Estimator 3/300, Train metric: 50.5340
Estimator 4/300, Train metric: 48.7228
Estimator 5/300, Train metric: 47.0657
Estimator 6/300, Train metric: 46.0698
Estimator 7/300, Train metric: 44.7027
Estimator 8/300, Train metric: 43.7969
Estimator 9/300, Train metric: 42.3529
Estimator 10/300, Train metric: 41.1388
Estimator 11/300, Train metric: 39.9451
Estimator 12/300, Train metric: 39.3936
Estimator 13/300, Train metric: 38.5708
Estimator 14/300, Train metric: 37.4268
Estimator 15/300, Train metric: 36.2130
Estimator 16/300, Train metric: 35.8212
Estimator 17/300, Train metric: 35.3588
Estimator 18/300, Train metric: 34.8089
Estimator 19/300, Train metric: 34.4374
Estimator 20/300, Train metric: 34.0438
Estimator 21/300, Train metric: 33.6476
Estimator 22/300, Train metric: 32.8975
Estimator 23/300, Train metric: 32.2363
Estimator 24/300, Train metric: 31.7545
Estimator 

Estimator 34/300, Train metric: 25.2632
Estimator 35/300, Train metric: 24.6503
Estimator 36/300, Train metric: 24.2793
Estimator 37/300, Train metric: 23.8609
Estimator 38/300, Train metric: 23.3188
Estimator 39/300, Train metric: 22.8892
Estimator 40/300, Train metric: 22.4073
Estimator 41/300, Train metric: 21.8657
Estimator 42/300, Train metric: 21.3036
Estimator 43/300, Train metric: 21.0520
Estimator 44/300, Train metric: 20.6351
Estimator 45/300, Train metric: 20.4684
Estimator 46/300, Train metric: 20.1000
Estimator 47/300, Train metric: 19.8026
Estimator 48/300, Train metric: 19.5589
Estimator 49/300, Train metric: 19.4191
Estimator 50/300, Train metric: 19.0989
Estimator 51/300, Train metric: 18.7039
Estimator 52/300, Train metric: 18.7178
Estimator 53/300, Train metric: 18.5688
Estimator 54/300, Train metric: 18.3391
Estimator 55/300, Train metric: 17.9783
Estimator 56/300, Train metric: 17.8861
Estimator 57/300, Train metric: 17.7476
Estimator 58/300, Train metric: 17.4496


Estimator 81/300, Train metric: 13.8087
Estimator 82/300, Train metric: 13.6064
Estimator 83/300, Train metric: 13.4929
Estimator 84/300, Train metric: 13.3332
Estimator 85/300, Train metric: 13.2440
Estimator 86/300, Train metric: 13.0907
Estimator 87/300, Train metric: 13.0373
Estimator 88/300, Train metric: 13.0103
Estimator 89/300, Train metric: 12.8905
Estimator 90/300, Train metric: 12.8355
Estimator 91/300, Train metric: 12.7364
Estimator 92/300, Train metric: 12.5994
Estimator 93/300, Train metric: 12.4648
Estimator 94/300, Train metric: 12.2250
Estimator 95/300, Train metric: 12.0125
Estimator 96/300, Train metric: 11.9530
Estimator 97/300, Train metric: 11.9207
Estimator 98/300, Train metric: 11.8422
Estimator 99/300, Train metric: 11.7715
Estimator 100/300, Train metric: 11.7421
Estimator 101/300, Train metric: 11.6483
Estimator 102/300, Train metric: 11.4578
Estimator 103/300, Train metric: 11.3104
Estimator 104/300, Train metric: 11.1709
Estimator 105/300, Train metric: 11

Estimator 119/300, Train metric: 10.2728
Estimator 120/300, Train metric: 10.2700
Estimator 121/300, Train metric: 10.1300
Estimator 122/300, Train metric: 10.0422
Estimator 123/300, Train metric: 9.9921
Estimator 124/300, Train metric: 9.9513
Estimator 125/300, Train metric: 9.9225
Estimator 126/300, Train metric: 9.8781
Estimator 127/300, Train metric: 9.8022
Estimator 128/300, Train metric: 9.8036
Estimator 129/300, Train metric: 9.6728
Estimator 130/300, Train metric: 9.6110
Estimator 131/300, Train metric: 9.5791
Estimator 132/300, Train metric: 9.5569
Estimator 133/300, Train metric: 9.5385
Estimator 134/300, Train metric: 9.4700
Estimator 135/300, Train metric: 9.4091
Estimator 136/300, Train metric: 9.4051
Estimator 137/300, Train metric: 9.2752
Estimator 138/300, Train metric: 9.1582
Estimator 139/300, Train metric: 9.1144
Estimator 140/300, Train metric: 9.1211
Estimator 141/300, Train metric: 9.0928
Estimator 142/300, Train metric: 9.0618
Estimator 143/300, Train metric: 9.0

Estimator 160/300, Train metric: 8.2039
Estimator 161/300, Train metric: 8.1449
Estimator 162/300, Train metric: 8.1309
Estimator 163/300, Train metric: 8.1263
Estimator 164/300, Train metric: 8.1300
Estimator 165/300, Train metric: 8.1188
Estimator 166/300, Train metric: 8.1135
Estimator 167/300, Train metric: 8.0945
Estimator 168/300, Train metric: 7.9866
Estimator 169/300, Train metric: 7.9883
Estimator 170/300, Train metric: 7.9698
Estimator 171/300, Train metric: 7.9719
Estimator 172/300, Train metric: 7.9138
Estimator 173/300, Train metric: 7.8723
Estimator 174/300, Train metric: 7.8243
Estimator 175/300, Train metric: 7.8278
Estimator 176/300, Train metric: 7.8433
Estimator 177/300, Train metric: 7.8214
Estimator 178/300, Train metric: 7.8264
Estimator 179/300, Train metric: 7.7773
Estimator 180/300, Train metric: 7.6972
Estimator 181/300, Train metric: 7.6182
Estimator 182/300, Train metric: 7.5688
Estimator 183/300, Train metric: 7.5420
Estimator 184/300, Train metric: 7.4684


Estimator 202/300, Train metric: 7.0839
Estimator 203/300, Train metric: 7.0691
Estimator 204/300, Train metric: 7.0687
Estimator 205/300, Train metric: 7.0643
Estimator 206/300, Train metric: 7.0131
Estimator 207/300, Train metric: 7.0047
Estimator 208/300, Train metric: 6.9527
Estimator 209/300, Train metric: 6.9280
Estimator 210/300, Train metric: 6.9172
Estimator 211/300, Train metric: 6.9223
Estimator 212/300, Train metric: 6.8699
Estimator 213/300, Train metric: 6.8710
Estimator 214/300, Train metric: 6.8053
Estimator 215/300, Train metric: 6.7838
Estimator 216/300, Train metric: 6.7737
Estimator 217/300, Train metric: 6.7405
Estimator 218/300, Train metric: 6.6536
Estimator 219/300, Train metric: 6.6429
Estimator 220/300, Train metric: 6.6378
Estimator 221/300, Train metric: 6.6014
Estimator 222/300, Train metric: 6.6011
Estimator 223/300, Train metric: 6.6056
Estimator 224/300, Train metric: 6.5337
Estimator 225/300, Train metric: 6.4972
Estimator 226/300, Train metric: 6.4960


Estimator 245/300, Train metric: 6.2135
Estimator 246/300, Train metric: 6.2100
Estimator 247/300, Train metric: 6.2147
Estimator 248/300, Train metric: 6.1478
Estimator 249/300, Train metric: 6.1009
Estimator 250/300, Train metric: 6.0913
Estimator 251/300, Train metric: 6.0789
Estimator 252/300, Train metric: 6.0865
Estimator 253/300, Train metric: 6.0385
Estimator 254/300, Train metric: 6.0455
Estimator 255/300, Train metric: 6.0139
Estimator 256/300, Train metric: 5.9809
Estimator 257/300, Train metric: 5.9833
Estimator 258/300, Train metric: 5.9267
Estimator 259/300, Train metric: 5.8744
Estimator 260/300, Train metric: 5.8320
Estimator 261/300, Train metric: 5.8273
Estimator 262/300, Train metric: 5.8217
Estimator 263/300, Train metric: 5.8125
Estimator 264/300, Train metric: 5.8159
Estimator 265/300, Train metric: 5.8148
Estimator 266/300, Train metric: 5.7728
Estimator 267/300, Train metric: 5.7383
Estimator 268/300, Train metric: 5.7390
Estimator 269/300, Train metric: 5.7183


Estimator 289/300, Train metric: 5.3629
Estimator 290/300, Train metric: 5.3620
Estimator 291/300, Train metric: 5.3140
Estimator 292/300, Train metric: 5.3226
Estimator 293/300, Train metric: 5.3074
Estimator 294/300, Train metric: 5.2770
Estimator 295/300, Train metric: 5.2490
Estimator 296/300, Train metric: 5.2060
Estimator 297/300, Train metric: 5.1671
Estimator 298/300, Train metric: 5.1673
Estimator 299/300, Train metric: 5.1422
Mean Squared Error: 381.83
         Mean    StdDev  Actual
0   90.790436  0.944250   73.07
1  144.716690  0.127969  141.67
2   98.323929  0.082583  110.44
3  267.103119  1.031815  275.01
4  165.928894  0.871511  202.57


In [27]:
generate_and_save_plots(predictions_PGBM_df, y_test, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


In [28]:
generate_and_save_crps_decomposition(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx
   Sheets: CRPS_Summary_PGBM, CRPS_Bins_PGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,PGBM,15.335806,8.265482,23.048132,30.119647,4


In [29]:
generate_and_save_pit_results(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


,PIT_Value
0,7.071659e-79
1,1.377516e-125
2,1.000000e+00
3,1.000000e+00
4,1.000000e+00
5,1.000000e+00
6,8.089476e-233
7,1.000000e+00
8,7.497806e-19
9,0.000000e+00


In [30]:
generate_and_save_calibration_curve(
    predictions_PGBM_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx",
    model_name="PGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/PGBM_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.340426,0.240426
2,0.2,0.340426,0.140426
3,0.3,0.340426,0.040426
4,0.4,0.340426,0.059574
5,0.5,0.340426,0.159574
6,0.6,0.340426,0.259574
7,0.7,0.361702,0.338298
8,0.8,0.361702,0.438298
9,0.9,0.361702,0.538298


# **Probabilistic Distribution using GPBoost (Treeffuser)**


In [31]:
best_params = get_best_model_params(best_scores_autosampler, 'GPBoost')

gpboost_score_engine = TreeBasedScoreModel(
    model_name='gpboost',
    model_params=best_params,
    seed=42,
    verbose=-1
)

# The fix is in the line below: '_score_model' has been corrected to 'score_model'.
model = Treeffuser(score_model=gpboost_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_GPBoost_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GPBoost_TF_df.head())


[1]	valid_0's l2: 2.17977
[2]	valid_0's l2: 2.08998
[3]	valid_0's l2: 2.09766
[4]	valid_0's l2: 2.06548
[5]	valid_0's l2: 2.03852
[6]	valid_0's l2: 2.03046
[7]	valid_0's l2: 2.03543
[8]	valid_0's l2: 2.03048
[9]	valid_0's l2: 2.03663
[10]	valid_0's l2: 1.9553
[11]	valid_0's l2: 1.95697
[12]	valid_0's l2: 1.93732
[13]	valid_0's l2: 1.86124
[14]	valid_0's l2: 1.86432
[15]	valid_0's l2: 1.79448
[16]	valid_0's l2: 1.69791
[17]	valid_0's l2: 1.68796
[18]	valid_0's l2: 1.63303
[19]	valid_0's l2: 1.57608
[20]	valid_0's l2: 1.57948
[21]	valid_0's l2: 1.58274
[22]	valid_0's l2: 1.5303
[23]	valid_0's l2: 1.5333
[24]	valid_0's l2: 1.53125
[25]	valid_0's l2: 1.46409
[26]	valid_0's l2: 1.44894
[27]	valid_0's l2: 1.39521
[28]	valid_0's l2: 1.39109
[29]	valid_0's l2: 1.39151
[30]	valid_0's l2: 1.39019
[31]	valid_0's l2: 1.38578
[32]	valid_0's l2: 1.37153
[33]	valid_0's l2: 1.37296
[34]	valid_0's l2: 1.37264
[35]	valid_0's l2: 1.32644
[36]	valid_0's l2: 1.26927
[37]	valid_0's l2: 1.26821
[38]	valid_0'

         Mean      StdDev  Actual
0  138.313769  240.509434   73.07
1  144.500987  125.601729  141.67
2  120.775547  270.882630  110.44
3  322.525204  137.911356  275.01
4  274.300086  176.525869  202.57


In [32]:
generate_and_save_plots(predictions_GPBoost_TF_df, y_test, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


In [33]:
generate_and_save_crps_decomposition(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx
   Sheets: CRPS_Summary_GPBoost, CRPS_Bins_GPBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,GPBoost,46.731715,24.835625,8.215621,30.119647,4


In [34]:
generate_and_save_pit_results(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


,PIT_Value
0,0.393090
1,0.491009
2,0.484782
3,0.365222
4,0.342245
5,0.585493
6,0.427454
7,0.375632
8,0.313518
9,0.397971


In [35]:
generate_and_save_calibration_curve(
    predictions_GPBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx",
    model_name="GPBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GPBoost_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.297872,0.102128
5,0.5,0.723404,0.223404
6,0.6,0.936170,0.336170
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using Gradient Boosting (Treeffuser)**

In [36]:
best_params = get_best_model_params(best_scores_autosampler, 'GradientBoosting')

gbr_score_engine = TreeBasedScoreModel(
    model_name='gradient_boosting',
    # FIX T1: the diffusion score model is fitted by least squares (Treeffuser's denoising
    # objective); the tuned GBR loss may be absolute_error/huber, so force squared_error here.
    model_params={**best_params, 'loss': 'squared_error'},
    seed=42,
)

model = Treeffuser(score_model=gbr_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_GBR_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_GBR_TF_df.head())


         Mean       StdDev  Actual
0  543.413519   966.056589   73.07
1  381.408634  1018.467336  141.67
2  476.436725  1054.546173  110.44
3  558.158373  1131.718547  275.01
4  621.810822  1122.539930  202.57


In [37]:
generate_and_save_plots(predictions_GBR_TF_df, y_test, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


In [38]:
generate_and_save_crps_decomposition(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="Gradient Boosting",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx
   Sheets: CRPS_Summary_Gradient Boosting, CRPS_Bins_Gradient Boosting


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,Gradient Boosting,296.242641,247.351435,-18.820824,30.119647,4


In [39]:
generate_and_save_pit_results(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="GBoosting",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


,PIT_Value
0,0.313175
1,0.406952
2,0.364271
3,0.401219
4,0.354397
5,0.448725
6,0.345118
7,0.388837
8,0.404939
9,0.404614


In [40]:
generate_and_save_calibration_curve(
    predictions_GBR_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx",
    model_name="GBoosting",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/GBoosting_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.659574,0.259574
5,0.5,1.000000,0.500000
6,0.6,1.000000,0.400000
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using Catboost (Treeffuser)**

In [41]:
best_params = get_best_model_params(best_scores_autosampler, 'CatBoost')

catboost_score_engine = TreeBasedScoreModel(
    model_name='catboost',
    model_params=best_params,
    seed=42,
)

model = Treeffuser(score_model=catboost_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_CatBoost_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_CatBoost_TF_df.head())

         Mean      StdDev  Actual
0  139.641507  252.868526   73.07
1  178.258679  169.212552  141.67
2  121.804399  139.546914  110.44
3  292.174655  118.480282  275.01
4  246.321995  150.706741  202.57


In [42]:
generate_and_save_plots(predictions_CatBoost_TF_df, y_test, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


In [43]:
generate_and_save_crps_decomposition(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_bins=N_BINS
)


✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx
   Sheets: CRPS_Summary_CatBoost, CRPS_Bins_CatBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,CatBoost,47.683142,31.000512,13.427875,30.119647,4


In [44]:
generate_and_save_pit_results(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


,PIT_Value
0,0.396173
1,0.414405
2,0.467547
3,0.442405
4,0.385789
5,0.554251
6,0.401341
7,0.412263
8,0.400429
9,0.447454


In [45]:
generate_and_save_calibration_curve(
    predictions_CatBoost_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx",
    model_name="CatBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/CatBoost_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.127660,0.272340
5,0.5,0.787234,0.287234
6,0.6,0.957447,0.357447
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using XGBoost (Treeffuser)**

In [46]:
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

xgb_score_engine = TreeBasedScoreModel(
    model_name='xgboost',
    model_params=best_params,
    seed=42,
    verbose=-1
)

model = Treeffuser(score_model=xgb_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_XGB_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_XGB_TF_df.head())

[0]	validation_0-rmse:1.46744


[1]	validation_0-rmse:1.42851


[2]	validation_0-rmse:1.33638


[3]	validation_0-rmse:1.27981


[4]	validation_0-rmse:1.22591


[5]	validation_0-rmse:1.16507


... (291 outputs left out)

[297]	validation_0-rmse:0.69466


[298]	validation_0-rmse:0.69494


[299]	validation_0-rmse:0.69515


         Mean      StdDev  Actual
0   64.970678  319.577162   73.07
1  170.926469  240.310175  141.67
2   98.541981  185.860713  110.44
3  410.515327  394.677623  275.01
4  352.315818  367.981194  202.57


In [47]:
generate_and_save_plots(predictions_XGB_TF_df, y_test, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


In [48]:
generate_and_save_crps_decomposition(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx
   Sheets: CRPS_Summary_XGBoost, CRPS_Bins_XGBoost


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGBoost,64.987192,40.496902,5.617485,30.119647,4


In [49]:
generate_and_save_pit_results(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


,PIT_Value
0,0.510110
1,0.451551
2,0.525521
3,0.365674
4,0.342026
5,0.516363
6,0.360007
7,0.467721
8,0.542343
9,0.463341


In [50]:
generate_and_save_calibration_curve(
    predictions_XGB_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx",
    model_name="XGBoost",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.191489,0.208511
5,0.5,0.680851,0.180851
6,0.6,0.978723,0.378723
7,0.7,1.000000,0.300000
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **Probabilistic Distribution using LGBM (Treeffuser)**

In [51]:
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

lgbm_score_engine = TreeBasedScoreModel(
    model_name='lightgbm',
    model_params=best_params,
    seed=42,
    verbose=-1
)

model = Treeffuser(score_model=lgbm_score_engine, seed=42)
model.fit(X_train, y_train)

n_samples_for_prediction = 200
y_samples = model.sample(X_test, n_samples=n_samples_for_prediction, verbose=False)

y_samples_obj = Samples(y_samples)
mean = y_samples_obj.sample_mean()
stddev = y_samples_obj.sample_std()

predictions_LGBM_TF_df = pd.DataFrame({
    'Mean': mean,
    'StdDev': stddev,
    'Actual': y_test
})

print(predictions_LGBM_TF_df.head())


         Mean      StdDev  Actual
0  127.385863  290.878219   73.07
1  186.888755  213.551877  141.67
2  114.019969  199.614591  110.44
3  380.332409  281.871546  275.01
4  281.562181  325.301354  202.57


In [52]:
generate_and_save_plots(predictions_LGBM_TF_df, y_test, "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx")

Excel file saved at ./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


In [53]:
generate_and_save_crps_decomposition(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx
   Sheets: CRPS_Summary_LightGBM, CRPS_Bins_LightGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LightGBM,52.534192,28.92127,6.497709,30.119647,4


In [54]:
generate_and_save_pit_results(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


,PIT_Value
0,0.425936
1,0.416153
2,0.492846
3,0.354331
4,0.404069
5,0.772865
6,0.436108
7,0.407274
8,0.346525
9,0.401184


In [55]:
generate_and_save_calibration_curve(
    predictions_LGBM_TF_df,
    y_test,
    "./drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx",
    model_name="LightGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob_TF.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.000000,0.200000
3,0.3,0.000000,0.300000
4,0.4,0.170213,0.229787
5,0.5,0.702128,0.202128
6,0.6,0.914894,0.314894
7,0.7,0.978723,0.278723
8,0.8,1.000000,0.200000
9,0.9,1.000000,0.100000


# **IBUG Installation**

In [56]:
# Clone the repository
!git clone https://github.com/jjbrophy47/ibug.git
%cd ibug

# Build extensions if needed
!python setup.py build_ext --inplace

# Install the package
!pip install .

# Verify installation
try:
    from ibug import IBUGWrapper
    print("IBUGWrapper imported successfully!")
except ImportError as e:
    print("Error importing IBUGWrapper:", e)

Cloning into 'ibug'...


remote: Enumerating objects: 1763, done.
remote: Counting objects: 100% (37/37), done.
remote: Compressing objects: 100% (31/31), done.


remote: Total 1763 (delta 18), reused 15 (delta 4), pack-reused 1726 (from 1)
Receiving objects: 100% (1763/1763), 1.31 MiB | 14.25 MiB/s, done.


Resolving deltas: 100% (967/967), done.


/kaggle/working/ibug


Compiling ibug/parsers/_tree32.pyx because it changed.
Compiling ibug/parsers/_tree64.pyx because it changed.
[1/2] Cythonizing ibug/parsers/_tree32.pyx


... (16 outputs left out)

  Preparing metadata (pyproject.toml) ... done


  Created wheel for ibug: filename=ibug-0.0.12-cp313-cp313-linux_x86_64.whl size=1581582 sha256=cfe3769796ade8749d28f9379221d32f22f1db5dbb9a4a2967dffe129233ee87
  Stored in directory: /tmp/pip-ephem-wheel-cache-4ub_o1wy/wheels/48/c2/c1/d39bc91440bc7e7da40b146580d8aeb84c5361fa857fe8037e
Successfully built ibug


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [ibug]


IBUGWrapper imported successfully!


In [57]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.3, random_state=42)  # FIX P2: seeded


# **Probabilistic Distributions using LGBM (IBUG)**

In [58]:
# y_train = y_train.to_numpy()
# y_val = y_val.to_numpy()

In [59]:
# Assume hyperparameter tuning has been done and best_params are obtained
best_params = get_best_model_params(best_scores_autosampler, 'LightGBM')

# --- Ensure IBUG compatibility ---
best_params['reg_alpha'] = 0
best_params['class_weight'] = None
best_params['boosting_type'] = 'gbdt'

# Initialize and train the LightGBM regressor using the best_params
model = LGBMRegressor(**best_params)
model.fit(X_train, y_train)

# Extend the trained model into a probabilistic predictor using IBUGWrapper
prob_model = IBUGWrapper().fit(model, X_train, y_train, X_val=X_val, y_val=y_val)

# Use the model to predict the probabilistic distribution on the test data.
location, scale = prob_model.pred_dist(X_test)

# Optionally, to also get the k-nearest training indices and their values used in modeling
location2, scale2, train_idxs, train_vals = prob_model.pred_dist(X_test, return_kneighbors=True)

dev = scale  # FIX P2: IBUG's scale is already a standard deviation (np.std of neighbours); was scale**0.5
# Create a DataFrame for clarity
predictions_LGBM_df = pd.DataFrame({
    'Mean': location,
    'StdDev': dev,
    'Actual': y_test,
})

print(predictions_LGBM_df.head())

         Mean     StdDev  Actual
0   86.780731  27.280817   73.07
1  110.669349  29.564255  141.67
2  101.337471  33.817797  110.44
3  274.747864  69.640450  275.01
4  227.657538  71.684631  202.57


In [60]:
%cd ..

/kaggle/working


In [61]:
%cd ..

/kaggle


In [62]:
generate_and_save_plots(predictions_LGBM_df, y_test, "./content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx")

Excel file saved at ./content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


In [63]:
generate_and_save_crps_decomposition(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx
   Sheets: CRPS_Summary_LGBM, CRPS_Bins_LGBM


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,LGBM,15.843094,2.733443,17.007974,30.119647,4


In [64]:
generate_and_save_pit_results(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   /content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


,PIT_Value
0,0.307631
1,0.852816
2,0.606098
3,0.501502
4,0.363180
5,0.837897
6,0.647864
7,0.354536
8,0.511539
9,0.752157


In [65]:
generate_and_save_calibration_curve(
    predictions_LGBM_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx",
    model_name="LGBM",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   /content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/LGBM_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.021277,0.178723
3,0.3,0.063830,0.236170
4,0.4,0.170213,0.229787
5,0.5,0.234043,0.265957
6,0.6,0.382979,0.217021
7,0.7,0.553191,0.146809
8,0.8,0.787234,0.012766
9,0.9,0.957447,0.057447


# **Probabilistic Distributions using XGBoost (IBUG)**

In [66]:
# Assume hyperparameter tuning has been done and best_params are obtained
best_params = get_best_model_params(best_scores_autosampler, 'XGBoost')

best_params['reg_alpha'] = 0
best_params['scale_pos_weight'] = 1
best_params['base_score'] = 0.5

# Initialize and train the XGBoost regressor using the best_params
model = XGBRegressor(**best_params)
model.fit(X_train, y_train)

# Extend the trained model into a probabilistic predictor using IBUGWrapper
prob_model = IBUGWrapper().fit(model, X_train, y_train, X_val=X_val, y_val=y_val)

# Use the model to predict the probabilistic distribution on the test data.
# This returns the predicted mean (location) and uncertainty (scale, variance proxy)
location, scale = prob_model.pred_dist(X_test)

# Optionally, to also get the k-nearest training indices and their values used in modeling
location, scale, train_idxs, train_vals = prob_model.pred_dist(X_test, return_kneighbors=True)

# Calculate the standard deviation from the variance (scale)
std_dev = scale  # FIX P2: IBUG's scale is already a standard deviation (np.std of neighbours); was scale ** 0.5

# Create a DataFrame for clarity
predictions_XGB_df = pd.DataFrame({
    'Mean': location,
    'StdDev': std_dev,
    'Actual': y_test
})

print(predictions_XGB_df.head())

         Mean     StdDev  Actual
0  101.665573  32.145248   73.07
1  149.188751  36.286827  141.67
2   83.764290  32.802998  110.44
3  231.103455  71.952583  275.01
4  170.003296  69.902168  202.57


In [67]:
generate_and_save_plots(predictions_XGB_df, y_test, "./content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx")  # FIX P3: was predictions_LGBM_df

Excel file saved at ./content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


In [68]:
generate_and_save_crps_decomposition(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_bins=N_BINS
)

✅ CRPS decomposition saved to:
/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx
   Sheets: CRPS_Summary_XGB, CRPS_Bins_XGB


,Model,Total_CRPS,Reliability,Resolution,Uncertainty,Num_Bins
0,XGB,16.699866,3.521935,16.93986,30.119647,4


In [69]:
generate_and_save_pit_results(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_bins=N_BINS
)

✅ PIT values, histogram, summary, and EMBEDDED plot saved to:
   /content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


,PIT_Value
0,0.186847
1,0.417926
2,0.791951
3,0.729140
4,0.679353
5,0.867481
6,0.372988
7,0.529298
8,0.544673
9,0.592737


In [70]:
generate_and_save_calibration_curve(
    predictions_XGB_df,
    y_test,
    "/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx",
    model_name="XGB",
    n_quantiles=10
)

✅ Calibration curve & metrics saved to Excel:
   /content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/XGB_Prob.xlsx


,Predicted_Quantile,Observed_Frequency,Absolute_Error
0,0.0,0.000000,0.000000
1,0.1,0.000000,0.100000
2,0.2,0.021277,0.178723
3,0.3,0.042553,0.257447
4,0.4,0.127660,0.272340
5,0.5,0.234043,0.265957
6,0.6,0.425532,0.174468
7,0.7,0.574468,0.125532
8,0.8,0.723404,0.076596
9,0.9,0.914894,0.014894


# **Matrix Evaulation**

In [71]:
import os

def _ensure_parent_dir(path):
    parent = os.path.dirname(path)
    if parent:
        os.makedirs(parent, exist_ok=True)
    return path

# Create a Pandas Excel writer using XlsxWriter as the engine.
folder_path = "/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis"
excel_path = os.path.join(folder_path, "/content/drive/MyDrive/Regression/water_quality_ewqi_uncertainty_analysis/probabilistic_distribution/Matrix Evaluation.xlsx")
with pd.ExcelWriter(_ensure_parent_dir(excel_path), engine='xlsxwriter') as writer:
    plot_metrics_and_save_to_excel(writer, predictions_NGB_df, y_test, 'NGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_PGBM_df, y_test, 'PGBM')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_df, y_test, 'LGBM')
    plot_metrics_and_save_to_excel(writer, predictions_XGB_df, y_test, 'XGBoost')
    plot_metrics_and_save_to_excel(writer, predictions_LGBM_TF_df, y_test, "LightGBM_TF")
    plot_metrics_and_save_to_excel(writer, predictions_XGB_TF_df, y_test, "XGBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_CatBoost_TF_df, y_test, "CatBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_GPBoost_TF_df, y_test, "GPBoost_TF")
    plot_metrics_and_save_to_excel(writer, predictions_GBR_TF_df, y_test, "GradientBoosting_TF")